<a href="https://colab.research.google.com/github/anisabendada-debug/uk-sme-lending-analysis/blob/main/sme_lending_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [77]:
   import pandas as pd

   lending = pd.read_csv("sme_lending.csv")
   rate = pd.read_csv("bank_rate.csv")

   print(lending.head())
   print(rate.head())


        Date  \
0  31 Aug 26   
1  31 Jul 26   
2  30 Jun 26   
3  31 May 26   
4  30 Apr 26   

   Monthly changes of monetary financial institutions' sterling and all foreign currency gross lending to small and medium sized enterprises (in sterling millions) seasonally adjusted \t\t\t\t\t\tRPMZM8B  
0                                               6003                                                                                                                                                         
1                                               6413                                                                                                                                                         
2                                               6022                                                                                                                                                         
3                                               5791                           

In [78]:
lending.columns = ["date", "sme_gross_lending"]

rate.columns = ["date", "bank_rate"]


In [79]:
print(lending.head())
print(rate.head())

        date  sme_gross_lending
0  31 Aug 26               6003
1  31 Jul 26               6413
2  30 Jun 26               6022
3  31 May 26               5791
4  30 Apr 26               6385
        date  bank_rate
0  30 Sep 26       3.75
1  31 Aug 26       3.75
2  31 Jul 26       3.75
3  30 Jun 26       3.75
4  31 May 26       3.75


In [80]:
lending["date"] = pd.to_datetime(lending["date"], dayfirst=True)
rate["date"] = pd.to_datetime(rate["date"], dayfirst=True)
lending["month"] =lending["date"].dt.to_period("M")

/tmp/ipykernel_2260/488811363.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  lending["date"] = pd.to_datetime(lending["date"], dayfirst=True)
/tmp/ipykernel_2260/488811363.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  rate["date"] = pd.to_datetime(rate["date"], dayfirst=True)


In [81]:
print(lending["date"].head())
print(rate["date"].head())


0   2026-08-31
1   2026-07-31
2   2026-06-30
3   2026-05-31
4   2026-04-30
Name: date, dtype: datetime64[ns]
0   2026-09-30
1   2026-08-31
2   2026-07-31
3   2026-06-30
4   2026-05-31
Name: date, dtype: datetime64[ns]


In [82]:
rate["month"] = rate["date"].dt.to.period("M")

In [83]:
rate["month"] = rate["date"].dt.to_period("M")

In [84]:
rate = rate.set_index("date").sort_index()
rate = rate.resample("MS").last().ffill().reset_index()
rate["month"] = rate["date"].dt.to_period("M")


In [85]:
data = lending[["month", "sme_gross_lending"]].merge(
    rate[["month", "bank_rate"]], on="month")
data = data[data["month"] >= "2015-01"]

print(data.shape)
print(data.isna().sum())
print(data.tail())

(140, 3)
month                0
sme_gross_lending    0
bank_rate            0
dtype: int64
       month  sme_gross_lending  bank_rate
135  2015-05               4728        0.5
136  2015-04               5012        0.5
137  2015-03               4843        0.5
138  2015-02               4694        0.5
139  2015-01               4742        0.5


In [86]:
   print(lending["month"].min(), rate["month"].min())


2015-01 2015-01


In [87]:
   data.to_csv("clean_data.csv", index=False)

In [88]:
insolv = pd.read_csv("insolvencies.csv")
print(insolv.shape)
print(insolv.columns)
print(insolv.head(10))

(320, 57)
Index(['period', 'year', 'month', 'total_ew_sa', 'compliq_ew_sa', 'cvl_ew_sa',
       'admin_ew_sa', 'total_ew_nsa', 'compliq_ew_nsa', 'cvl_ew_nsa',
       'admin_ew_nsa', 'cva_ew_nsa', 'rec_ew_nsa', 'mor_ew_nsa',
       'restplan_ew_nsa', 'tot_rate_ew', 'compliq_rate_ew', 'cvl_rate_ew',
       'admin_rate_ew', 'cva_rate_ew', 'rec_rate_ew', 'adcvl_ew_nsa',
       'adcvl_rate_ew', 'cvl_with_bulk_ew', 'cvl_with_bulk_rate_ew',
       'total_sc_nsa', 'compliq_sc_nsa', 'cvl_sc_nsa', 'admin_sc_nsa',
       'cva_sc_nsa', 'rec_sc_nsa', 'mor_sc_nsa', 'restplan_sc_nsa',
       'tot_rate_sc', 'compliq_rate_sc', 'cvl_rate_sc', 'admin_rate_sc',
       'cva_rate_sc', 'rec_rate_sc', 'adcvl_sc_nsa', 'adcvl_rate_sc',
       'total_ni_nsa', 'compliq_ni_nsa', 'cvl_ni_nsa', 'admin_ni_nsa',
       'cva_ni_nsa', 'rec_ni_nsa', 'mor_ni_nsa', 'restplan_ni_nsa',
       'tot_rate_ni', 'compliq_rate_ni', 'cvl_rate_ni', 'admin_rate_ni',
       'cva_rate_ni', 'rec_rate_ni', 'adcvl_ni_nsa', 'adcvl_rate_ni'

In [89]:
insolv = insolv[["period", "total_ew_sa"]].copy()
insolv.columns = ["month","insolvencies"]
insolv["month"] = pd.to_datetime(insolv["month"]).dt.to_period("M")
insolv["insolvencies"] = pd.to_numeric(insolv["insolvencies"], errors="coerce")
print(insolv.tail())

       month  insolvencies
315  2026-04          2083
316  2026-05          1860
317  2026-06          1850
318  2026-07          1934
319  2026-08          1946


In [90]:
data = data.merge(insolv, on="month", how="left")
print(data.shape)
print(data.isna().sum())
print(data.tail())

(140, 4)
month                0
sme_gross_lending    0
bank_rate            0
insolvencies         0
dtype: int64
       month  sme_gross_lending  bank_rate  insolvencies
135  2015-05               4728        0.5          1209
136  2015-04               5012        0.5          1235
137  2015-03               4843        0.5          1214
138  2015-02               4694        0.5          1301
139  2015-01               4742        0.5          1129


In [91]:
data = data.sort_values("month").reset_index(drop=True)
print(data.head())
data.to_csv("clean_data.csv", index=False)

     month  sme_gross_lending  bank_rate  insolvencies
0  2015-01               4742        0.5          1129
1  2015-02               4694        0.5          1301
2  2015-03               4843        0.5          1214
3  2015-04               5012        0.5          1235
4  2015-05               4728        0.5          1209


In [92]:
data.to_csv("clean_data.csv", index=False)